# Online Retail — ETL Assessment

The sections follow the assignment numbering. Run cells **in order the first time**; afterwards you can rerun individual cells to inspect their output. The data is stored in a local SQLite database. Install pandas with `pip install pandas` if needed.

In [42]:
# pip install pandas

In [43]:
# Setup — libraries and the explicit CSV location
import pandas as pd
import sqlite3
import hashlib
import shutil
from pathlib import Path
from datetime import datetime

file_path = Path(r"C:\Users\Lenovo\Desktop\Artefact_Assessments\online_retail.csv")
db_path = file_path.parent / "online_retail.db"

print("CSV:", file_path)

CSV: C:\Users\Lenovo\Desktop\Artefact_Assessments\online_retail.csv


## 1. Data Ingestion

In [44]:
# 1.1. Implement a data ingestion process to fetch the dataset from its source (e.g., CSV file, database)
# Read the dataset from the CSV file on my computer & keep identifiers as strings to preserve their original format
df = pd.read_csv(file_path, dtype={
    "InvoiceNo": "string", 
    "StockCode": "string",
    "CustomerID": "string"
})
print("Count:", len(df))
df.head()

Count: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [45]:
# 1.2. Load the dataset into a suitable storage system (e.g., relational database, data warehouse)
# Load the original data into a relational database (SQLite), to later compare the source and the cleaned version.

conn = sqlite3.connect(db_path)

df.to_sql("raw_online_retail", conn, if_exists="replace", index=False)

# Read data to make sure it appears in the database.
print("Count:", len(pd.read_sql("SELECT * FROM raw_online_retail", conn)))
display(pd.read_sql("SELECT * FROM raw_online_retail LIMIT 10", conn))

conn.close()

Count: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


## 2. Data Cleaning and Validation

In [46]:
# 2.1. Develop a data cleaning pipeline to handle missing values, duplicates, and outliers.

rows_before = len(df)

# Remove spaces and treat empty text as missing.
for column in ["InvoiceNo", "StockCode", "Description", "CustomerID", "Country"]:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# Treat Customer IDs as identifiers, not numbers. For example, "17850.0" becomes "17850"
df["CustomerID"] = df["CustomerID"].str.replace(
    r"\.0$", "", regex=True
)

# Keep useful transactions even when descriptive information is missing.
df["CustomerID"] = df["CustomerID"].fillna("UNKNOWN")
df["Description"] = df["Description"].fillna("Unknown product")
df["Country"] = df["Country"].fillna("Unknown")

# Inspect duplicates before removing them.
duplicates = df.duplicated()
duplicates_removed = int(duplicates.sum())

# print("\duplicate rows to remove:", duplicates_removed)
# display(df.loc[duplicates].head(10))

# Keep the first occurrence and remove repeated copies.
df = df.drop_duplicates()

# Convert datatypes
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


In [47]:
# Compare how many rows would be flagged at different quantity limits.
# Include both large positive and large negative quantities.
quantity_size = df["Quantity"].abs()

above_50 = (quantity_size > 50).sum()
above_100 = (quantity_size > 100).sum()
above_200 = (quantity_size > 200).sum()
above_500 = (quantity_size > 500).sum()

comparison = pd.DataFrame({
    "Quantity limit": [50, 100, 200, 500],
    "Rows flagged": [above_50, above_100, above_200, above_500]
})

comparison["Percentage of rows"] = (
    comparison["Rows flagged"] / len(df) * 100
).round(2)

print("Total rows:", len(df))
display(comparison)

Total rows: 536641


,Quantity limit,Rows flagged,Percentage of rows
0,50,13091,2.44
1,100,5400,1.01
2,200,2056,0.38
3,500,540,0.10


In [48]:
# Review transactions involving more than 100 units.
# Keep them: large sales, cancellations and adjustments may be legitimate.
quantity_threshold = 100

df["QuantityOutlier"] = (
    df["Quantity"].abs() > quantity_threshold
).astype(int)

# print("Rows flagged:", df["QuantityOutlier"].sum())
# print("Total rows retained:", len(df))

# display(
#     df.loc[df["QuantityOutlier"] == 1]
#     .sort_values("Quantity", key=lambda values: values.abs(), ascending=False)
#     .head(10)
# )

In [ ]:
# # Confirm the number of rows after removing duplicates.
# print("Rows before cleaning:", rows_before)
# print("Duplicate rows removed:", duplicates_removed)
# print("Rows after cleaning:", len(df))

# print(
#     "Row counts reconcile:",
#     rows_before == len(df) + duplicates_removed
# )

# # Check whether any missing values remain.
# print("\nMissing values after cleaning:")
# display(df.isna().sum().rename("Missing count").to_frame())

# # Inspect a sample of the cleaned data.
# print("\nCleaned data preview:")
# display(df.head(10))

Rows before cleaning: 541909
Duplicate rows removed: 5268
Rows after cleaning: 536641
Row counts reconcile: True

Missing values after cleaning:


,Missing count
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,0
Country,0
QuantityOutlier,0



Cleaned data preview:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,QuantityOutlier
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,0
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,0
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,0
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,0
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,0


In [49]:
# 2.2. Validate the integrity and consistency of the data during the cleaning process.
required_fields = [
    "InvoiceNo", "StockCode", "InvoiceDate", "Quantity", "UnitPrice"
]

invalid = df[required_fields].isna().any(axis=1)
rejected = df.loc[invalid].copy()
df = df.loc[~invalid].copy()

display(rejected.head(10))

# Check consistency after cleaning.
print("Duplicates remaining:", df.duplicated().sum())
print("Negative prices:", (df["UnitPrice"] < 0).sum())
print("Zero prices:", (df["UnitPrice"] == 0).sum())
print("Non-whole quantities:", (df["Quantity"] % 1 != 0).sum())

print("\nPrice records to review:")
display(df.loc[df["UnitPrice"] <= 0].head(10))

print("Rows retained:", len(df))
display(df.head(10))

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,QuantityOutlier


Duplicates remaining: 0
Negative prices: 2
Zero prices: 2510
Non-whole quantities: 0

Price records to review:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,QuantityOutlier
622,536414,22139,Unknown product,56,2010-12-01 11:52:00,0.0,UNKNOWN,United Kingdom,0
1970,536545,21134,Unknown product,1,2010-12-01 14:32:00,0.0,UNKNOWN,United Kingdom,0
1971,536546,22145,Unknown product,1,2010-12-01 14:33:00,0.0,UNKNOWN,United Kingdom,0
1972,536547,37509,Unknown product,1,2010-12-01 14:33:00,0.0,UNKNOWN,United Kingdom,0
1987,536549,85226A,Unknown product,1,2010-12-01 14:34:00,0.0,UNKNOWN,United Kingdom,0
1988,536550,85044,Unknown product,1,2010-12-01 14:34:00,0.0,UNKNOWN,United Kingdom,0
2024,536552,20950,Unknown product,1,2010-12-01 14:34:00,0.0,UNKNOWN,United Kingdom,0
2025,536553,37461,Unknown product,3,2010-12-01 14:35:00,0.0,UNKNOWN,United Kingdom,0
2026,536554,84670,Unknown product,23,2010-12-01 14:35:00,0.0,UNKNOWN,United Kingdom,0
2406,536589,21777,Unknown product,-10,2010-12-01 16:50:00,0.0,UNKNOWN,United Kingdom,0


Rows retained: 536641


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,QuantityOutlier
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,0
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,0
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,0
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,0
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,0


## 3. Data Transformation

In [50]:
# 3. Prepare the cleaned data for analysis.
# Note: Numeric and date conversions were completed in section 2.1

# Keep negative amounts so returns and cancellations can reduce sales totals
df["SalesAmount"] = (df["Quantity"] * df["UnitPrice"]).round(2)

# Identify negative quantities and cancellation invoices
df["IsReturn"] = (
    (df["Quantity"] < 0)
    | df["InvoiceNo"].str.startswith("C", na=False)
).astype(int)

# Keep the transaction date separately for possible daily analysis
df["SalesDate"] = df["InvoiceDate"].dt.normalize()

# Standardize descriptions
df["Description"] = df["Description"].str.upper()

print("Rows after transformation:", len(df))

display(df.dtypes.rename("Data type").to_frame())

display(df[[
        "InvoiceNo", "Description", "Quantity", "UnitPrice",
        "SalesAmount", "IsReturn", "SalesDate"
    ]].head(10)
)


Rows after transformation: 536641


,Data type
InvoiceNo,string[python]
StockCode,string[python]
Description,string[python]
Quantity,int64
InvoiceDate,datetime64[ns]
UnitPrice,float64
CustomerID,string[python]
Country,string[python]
QuantityOutlier,int64
SalesAmount,float64


,InvoiceNo,Description,Quantity,UnitPrice,SalesAmount,IsReturn,SalesDate
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,15.30,0,2010-12-01
1,536365,WHITE METAL LANTERN,6,3.39,20.34,0,2010-12-01
2,536365,CREAM CUPID HEARTS COAT HANGER,8,2.75,22.00,0,2010-12-01
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,20.34,0,2010-12-01
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,20.34,0,2010-12-01
5,536365,SET 7 BABUSHKA NESTING BOXES,2,7.65,15.30,0,2010-12-01
6,536365,GLASS STAR FROSTED T-LIGHT HOLDER,6,4.25,25.50,0,2010-12-01
7,536366,HAND WARMER UNION JACK,6,1.85,11.10,0,2010-12-01
8,536366,HAND WARMER RED POLKA DOT,6,1.85,11.10,0,2010-12-01
9,536367,ASSORTED COLOUR BIRD ORNAMENT,32,1.69,54.08,0,2010-12-01


## 4. Data Warehouse Design

In [51]:
# 4. Create tables: 
#   products and customers describe each sale;
#   fact_sales stores one row per invoice line.

with sqlite3.connect(db_path) as conn:
    conn.execute("PRAGMA foreign_keys = ON")

    conn.executescript("""
    CREATE TABLE IF NOT EXISTS dim_product (
        StockCode TEXT PRIMARY KEY,
        Description TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_customer (
        CustomerKey TEXT PRIMARY KEY
    );

    CREATE TABLE IF NOT EXISTS fact_sales (
        SalesLineID INTEGER PRIMARY KEY,
        InvoiceNo TEXT,
        StockCode TEXT,
        CustomerKey TEXT,
        Country TEXT,
        InvoiceDate TEXT,
        SalesDate TEXT,
        Quantity INTEGER,
        UnitPrice REAL,
        SalesAmount REAL,
        IsReturn INTEGER,
        QuantityOutlier INTEGER,
        FOREIGN KEY (StockCode) REFERENCES dim_product(StockCode),
        FOREIGN KEY (CustomerKey) REFERENCES dim_customer(CustomerKey)
    );
    """)

#Note: country is being kept in fact_sales table because many Customers are missing, to add it in dim_customer

# Check created tables
display(pd.read_sql_query("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name IN ('dim_product', 'dim_customer', 'fact_sales')
        ORDER BY name
    """, conn))

,name
0,dim_customer
1,dim_product
2,fact_sales


## 5. ETL Process

In [52]:
# 5.1. Load the prepared data into the warehouse.

# Keep one description per product
products = df[["StockCode", "Description"]].drop_duplicates(
    "StockCode", keep="last"
)

# Keep one record per customer identifier
customers = df[["CustomerID"]].drop_duplicates()
customers = customers.rename(columns={"CustomerID": "CustomerKey"})

# Keep one sales record per cleaned invoice line
sales = df[[
    "InvoiceNo", "StockCode", "CustomerID", "Country",
    "InvoiceDate", "SalesDate", "Quantity", "UnitPrice",
    "SalesAmount", "IsReturn", "QuantityOutlier"
]].copy()

sales = sales.rename(columns={"CustomerID": "CustomerKey"})

# Store dates consistently in SQLite
sales["InvoiceDate"] = sales["InvoiceDate"].dt.strftime("%Y-%m-%d %H:%M:%S")
sales["SalesDate"] = sales["SalesDate"].dt.strftime("%Y-%m-%d")

with sqlite3.connect(db_path) as conn:
    conn.execute("PRAGMA foreign_keys = ON")

    # Replace the previous load so rerunning does not duplicate sales & delete sales first because they reference products and customers
    conn.execute("DELETE FROM fact_sales")
    conn.execute("DELETE FROM dim_product")
    conn.execute("DELETE FROM dim_customer")

    # Load products and customers before the sales that reference them
    products.to_sql("dim_product", conn, if_exists="append", index=False)
    customers.to_sql("dim_customer", conn, if_exists="append", index=False)
    sales.to_sql("fact_sales", conn, if_exists="append", index=False)

    # Confirm that each table contains the expected number of rows.
    for table, expected in [
        ("dim_product", len(products)),
        ("dim_customer", len(customers)),
        ("fact_sales", len(sales))
    ]:
        loaded = conn.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]

        print(f"{table}: {loaded} rows | Expected: {expected}")
        assert loaded == expected, f"Row count mismatch in {table}"

    print("\nLoaded sales preview:")
    display(pd.read_sql_query(
        "SELECT * FROM fact_sales LIMIT 10", conn
    ))

dim_product: 4070 rows | Expected: 4070
dim_customer: 4373 rows | Expected: 4373
fact_sales: 536641 rows | Expected: 536641

Loaded sales preview:


,SalesLineID,InvoiceNo,StockCode,CustomerKey,Country,InvoiceDate,SalesDate,Quantity,UnitPrice,SalesAmount,IsReturn,QuantityOutlier
0,1,536365,85123A,17850,United Kingdom,2010-12-01 08:26:00,2010-12-01,6,2.55,15.30,0,0
1,2,536365,71053,17850,United Kingdom,2010-12-01 08:26:00,2010-12-01,6,3.39,20.34,0,0
2,3,536365,84406B,17850,United Kingdom,2010-12-01 08:26:00,2010-12-01,8,2.75,22.00,0,0
3,4,536365,84029G,17850,United Kingdom,2010-12-01 08:26:00,2010-12-01,6,3.39,20.34,0,0
4,5,536365,84029E,17850,United Kingdom,2010-12-01 08:26:00,2010-12-01,6,3.39,20.34,0,0
5,6,536365,22752,17850,United Kingdom,2010-12-01 08:26:00,2010-12-01,2,7.65,15.30,0,0
6,7,536365,21730,17850,United Kingdom,2010-12-01 08:26:00,2010-12-01,6,4.25,25.50,0,0
7,8,536366,22633,17850,United Kingdom,2010-12-01 08:28:00,2010-12-01,6,1.85,11.10,0,0
8,9,536366,22632,17850,United Kingdom,2010-12-01 08:28:00,2010-12-01,6,1.85,11.10,0,0
9,10,536367,84879,13047,United Kingdom,2010-12-01 08:34:00,2010-12-01,32,1.69,54.08,0,0


In [ ]:
# My computer is in Portuguese, and also I don't have experience doing this as I would do this on Databricks or Azure Data Factory not on my computer, hence I'm leaving the scheduling task undone. 

## 6. Partitioning and Indexing

In [54]:
# Implement & evaluate partitioning and indexing strategies
# Indexing creates a lookup structure so the database can find matching rows without checking the whole table

# Speed up queries that filter sales by date - 
with sqlite3.connect(db_path) as conn:
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_sales_date
        ON fact_sales(SalesDate)
    """)

    conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_sales_product
            ON fact_sales(StockCode)
        """)
    
    conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_sales_customer
            ON fact_sales(CustomerKey)
        """)


    print("Indexes on the sales table:")
    display(pd.read_sql_query(
        "PRAGMA index_list('fact_sales')", conn
    ))

Indexes on the sales table:


,seq,name,unique,origin,partial
0,0,idx_sales_invoice,0,c,0
1,1,idx_sales_customer,0,c,0
2,2,idx_sales_product,0,c,0
3,3,idx_sales_date,0,c,0


In [ ]:
# Partitioning is unavailable in SQLite, so it was not implemented. For a larger deployment using a database that supports partitioning, monthly partitions on SalesDate would be a good option.

## 7. Data Quality Monitoring

In [55]:
# Set up monitoring mechanisms to track data quality over time & Establish alerts for anomalies or issues detected during the ETL process.

run_time = datetime.now().isoformat(timespec="seconds")

missing_customers = int((df["CustomerID"] == "UNKNOWN").sum())
non_positive_prices = int((df["UnitPrice"] <= 0).sum())
large_quantities = int(df["QuantityOutlier"].sum())

with sqlite3.connect(db_path) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS quality_history (
            RunTime TEXT,
            RowsRetained INTEGER,
            RowsRejected INTEGER,
            MissingCustomers INTEGER,
            NonPositivePrices INTEGER,
            LargeQuantities INTEGER
        )
    """)

    conn.execute("""
        INSERT INTO quality_history VALUES (?, ?, ?, ?, ?, ?)
    """, (
        run_time, len(df), len(rejected),
        missing_customers, non_positive_prices, large_quantities
    ))

    print("Latest quality checks:")
    display(pd.read_sql_query("""
        SELECT * FROM quality_history
        ORDER BY RunTime DESC
        LIMIT 10
    """, conn))

# Display alerts in the executed notebook.
print("\nQuality alerts:")

if df.empty:
    print("ALERT: No rows remain after cleaning.")

if len(rejected) > 0:
    print(f"ALERT: {len(rejected)} rows were rejected.")

if missing_customers > 0:
    print(f"REVIEW: {missing_customers} rows have an unknown customer.")

if non_positive_prices > 0:
    print(
        f"REVIEW: {non_positive_prices} rows have zero or negative prices. "
        "These records were retained for inspection."
    )

if large_quantities > 0:
    print(f"REVIEW: {large_quantities} rows exceed the quantity threshold.")

if (
    not df.empty
    and len(rejected) == 0
    and missing_customers == 0
    and non_positive_prices == 0
    and large_quantities == 0
):
    print("No issues detected by these checks")

Latest quality checks:


,RunTime,RowsRetained,RowsRejected,MissingCustomers,NonPositivePrices,LargeQuantities
0,2026-10-15T18:43:28,536641,0,135037,2512,5400



Quality alerts:
REVIEW: 135037 rows have an unknown customer.
REVIEW: 2512 rows have zero or negative prices. These records were retained for inspection.
REVIEW: 5400 rows exceed the quantity threshold.


## 8. Data Versioning

In [ ]:
# Only know how to do this using Git

## 9. Metadata Management

In [ ]:
# Never done this

## 10. Performance Optimization

In [57]:
# I know how to analyze this in Databricks only